# GMNS+ Training Guide v.6 — Traffic District 92 worked example
- Follow Steps 1-6 to build a **walk + transit** network and compare TAZ accessibility under free-flow and demand-dependent travel times.

- Run the notebook from this folder. Install the listed packages in Setup and restart the kernel.

- Inputs are stored in `raw_data/`. Generated files are saved in `step1_data/`, `step2_networks/`, `step3_zones/`, `step4_merge/` and `step6_accessibility/`.

- Distances are metres, speeds km/h, and travel times minutes.


## Setup
Install these versions once in the selected environment, then restart the kernel.


In [ ]:
# Run once if needed, then restart the kernel.

# Network conversion, zone connectors and routing.
# %pip install osm2gmns==1.0.1 gtfs2gmns==0.8.0 gmns-ready==0.1.2 DTALite==0.8.1

# Data processing, spatial analysis and visualization.
# %pip install geopandas==1.1.4 pandas==2.3.3 numpy==2.4.6 scipy==1.17.1 shapely==2.1.2 networkx==3.1
# %pip install pyproj==3.7.2 matplotlib==3.10.9 folium==0.20.0 geopy==2.4.1 contextily==1.7.1

# Transfer/merge helpers and table conversion.
# %pip install git+https://github.com/Kim-yongki/osu_gmns
# %pip install git+https://github.com/Kim-yongki/table2gmns.git


In [ ]:
# Folder paths and temporary working-directory changes.
from pathlib import Path
from contextlib import chdir
import os
import shutil

# Data processing, network analysis and visualization.
import contextily as ctx
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import networkx as nx
from IPython.display import display

# GMNS conversion, zone connectors, network merging and routing.
import osm2gmns as og
import table2gmns as tg
from gtfs2gmns import gtfs2gmns
import gmns_ready as gr
from gmns_ready.build_network import build_network
from gmns_ready.quick_check import quick_check
from osu_gmns import network_update, transfer_connector_builder, network_merge
import DTALite as dta

# Reset generated results before a full run.
# Delete only the listed step folders; retain raw_data and the notebook.
REMOVE = True
if REMOVE:
    import stat

    # Clear read-only attributes if they prevent deletion.
    def unlock(func, path, _):
        os.chmod(path, stat.S_IWRITE)
        func(path)

    # Remove each generated folder together with its contents.
    for folder in ["step1_data", "step2_networks", "step3_zones", "step4_merge", "step6_accessibility"]:
        if Path(folder).exists():
            shutil.rmtree(folder, onerror=unlock)


## Step 1: Data Acquisition and Preparation
### 1.1. Data Acquisition
Use the OSM walking extract, COTA GTFS and TAZ data in `raw_data/`. The bicycle LTS shapefile is used only in the separate `table2gmns` example at the end of Step 2.

### 1.2. Data Preparation
Select the 30 TAZs in Traffic District 92


In [ ]:
# Prepare the Step 1 output folder.
Path("step1_data").mkdir(parents=True, exist_ok=True)

# Select the TAZs in Traffic District 92.
taz = gpd.read_file("raw_data/TAZ20_OfficialMTP2024_Forecasts/TAZ20_OfficialMTP2024_Forecasts.shp")
taz = taz[taz["TraffDist"] == "92"]

# Retain the source TAZ identifier and save polygons in a metric CRS.
taz["TAZ"] = taz["TAZ2020"].astype(int)
taz = taz.to_crs("EPSG:26917")
taz.to_file("step1_data/taz.gpkg")


Prepare Saturday transit services for the walking + transit analysis.

In [ ]:
# Prepare a separate GTFS folder for regular Saturday services.
Path("step1_data/gtfs_saturday").mkdir(parents=True, exist_ok=True)

# Select services marked as operating on Saturdays.
calendar = pd.read_csv("raw_data/cota.gtfs/calendar.txt")
calendar = calendar[calendar["saturday"] == 1]

# Keep trips belonging to those services.
trips = pd.read_csv("raw_data/cota.gtfs/trips.txt", dtype=str)
trips = trips[trips["service_id"].isin(calendar["service_id"])]

# Keep stop-time records for the selected trips.
stop_times = pd.read_csv("raw_data/cota.gtfs/stop_times.txt", dtype=str)
stop_times = stop_times[stop_times["trip_id"].isin(trips["trip_id"])]

# Copy shared agency, stop and route tables.
shutil.copy2("raw_data/cota.gtfs/agency.txt", "step1_data/gtfs_saturday/agency.txt")
shutil.copy2("raw_data/cota.gtfs/stops.txt", "step1_data/gtfs_saturday/stops.txt")
shutil.copy2("raw_data/cota.gtfs/routes.txt", "step1_data/gtfs_saturday/routes.txt")

# Save the filtered tables; Step 2 selects the 14:00-16:00 time window.
trips.to_csv("step1_data/gtfs_saturday/trips.txt", index=False)
stop_times.to_csv("step1_data/gtfs_saturday/stop_times.txt", index=False)


### 1.3. Multimodal Representation Selection
Keep walking and transit in separate node/link tables. Step 4 creates transfers between them and combines them into one walk + transit network.


## Step 2: Convert Network Data to GMNS+ Format
Convert each source to `node.csv` and `link.csv`.

**OSM walking network.** Read the local OSM extract and select the study-area links and their endpoint nodes.


In [ ]:
# Convert OSM walking links to GMNS node and link tables.
Path("step2_networks/walk").mkdir(parents=True, exist_ok=True)
net = og.getNetFromFile("raw_data/planet_-83.093_39.983_7782789b.osm.pbf", mode_types="walk")
og.fillLinkAttributesWithDefaultValues(net, default_capacity=True)
og.outputNetToCSV(net, output_folder="step2_networks/walk")

# Read the exported tables and select links around the study district.
walk_node = pd.read_csv("step2_networks/walk/node.csv")
walk_link = pd.read_csv("step2_networks/walk/link.csv")
taz = gpd.read_file("step1_data/taz.gpkg")
taz = taz.to_crs("EPSG:4326")
network_geometry = taz.geometry.union_all().buffer(0.001)  # A small buffer in degrees includes links just outside the TAZ boundary.
geometry = gpd.GeoSeries.from_wkt(walk_link["geometry"], crs="EPSG:4326")
walk_link = walk_link[geometry.intersects(network_geometry)].copy()

# Retain only nodes referenced by the selected links.
used_nodes = pd.concat([walk_link["from_node_id"], walk_link["to_node_id"]]).unique()
walk_node = walk_node[walk_node["node_id"].isin(used_nodes)]

# Prepare the network for routing and connector generation.
# Keep the largest strongly connected component so retained nodes are mutually reachable.
walk_graph = nx.from_pandas_edgelist(walk_link, "from_node_id", "to_node_id", create_using=nx.DiGraph())
largest_component = max(nx.strongly_connected_components(walk_graph), key=len)
walk_link = walk_link[walk_link["from_node_id"].isin(largest_component) & walk_link["to_node_id"].isin(largest_component)].copy()
walk_node = walk_node[walk_node["node_id"].isin(largest_component)].copy()

# Use walking speeds and a uniform link type for connector selection.
walk_link["free_speed"] = 4.828032  # 3 mph expressed in km/h.
walk_link["link_type"] = 1  # Avoid prioritizing one source road class over another.

# Replace the initial exports with the prepared walking network.
walk_node.to_csv("step2_networks/walk/node.csv", index=False)
walk_link.to_csv("step2_networks/walk/link.csv", index=False)


**Transit network.** Convert Saturday services for 14:00–16:00.


In [ ]:
# Convert Saturday transit services for the 14:00-16:00 window.
Path("step2_networks/transit").mkdir(parents=True, exist_ok=True)
gtfs2gmns(input_path="step1_data/gtfs_saturday", output_path="step2_networks/transit", time_period="1400_1600")

# Prepare the travel-time field and illustrative assignment capacity.
transit_link = pd.read_csv("step2_networks/transit/link.csv")
transit_link = transit_link.rename(columns={"VDF_fftt": "vdf_fftt"}) # Rename `VDF_fftt` to `vdf_fftt`

# Set an illustrative values for zone connectors.
transit_link['capacity'] = 3000 # capacity for zone connectors (default is 99999).
transit_link["free_speed"] = transit_link["free_speed"].clip(upper=120) # maximum free speed (default raises errors in step 5).

# Convert stop-to-stop transfer times from hours to minutes.
is_stop_transfer = transit_link["link_type_name"] == "transferring_links"
transit_link.loc[is_stop_transfer, "vdf_fftt"] *= 60

# Save the prepared transit links for network merging.
transit_link.to_csv("step2_networks/transit/link.csv", index=False)


**Walking and transit networks.** The black outline marks Traffic District 92; transit retains its wider service area.


In [ ]:
# Use a common metric CRS for the district boundary and network maps.
plot_taz = gpd.read_file("step1_data/taz.gpkg").to_crs("EPSG:26917")

# Draw the walking and transit networks side by side.
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, mode, title, color in zip(axes, ["walk", "transit"], ["Walking links", "Transit links"], ["#3B79A5", "#C56E3D"]):
    # Load each exported link table and convert its WKT geometry.
    plot_links = pd.read_csv(f"step2_networks/{mode}/link.csv", low_memory=False)
    plot_links = gpd.GeoDataFrame(plot_links, geometry=gpd.GeoSeries.from_wkt(plot_links["geometry"], crs="EPSG:4326"), crs="EPSG:4326").to_crs("EPSG:26917")

    # Show the links and district outline without clipping the transit service area.
    plot_links.plot(ax=ax, color=color, linewidth=0.55, alpha=0.8)
    plot_taz.dissolve().boundary.plot(ax=ax, color="#222222", linewidth=1.1)
    ax.set_title(f"{title}\n{len(plot_links):,} directed links", fontsize=12)
    ax.set_aspect("equal")
    ax.set_axis_off()

# Add the shared legend and display the figure.
fig.legend(handles=[Line2D([0], [0], color="#222222", lw=1.1, label="Traffic District 92")], loc="lower center", frameon=False)
fig.suptitle("Step 2 | Converted modal networks", fontsize=15)
fig.tight_layout(rect=(0, 0.05, 1, 0.96))
display(fig)
plt.close(fig)


**Optional: bicycle conversion with `table2gmns`.** This network is not used in Steps 3–6. The cell below demonstrates how to preprocess a source data table and convert it to GMNS format.

In [ ]:
# Select LTS bicycle features around the study district.
taz = gpd.read_file("step1_data/taz.gpkg").to_crs("EPSG:4326")
network_geometry = taz.geometry.union_all().buffer(0.001)  # A small buffer in degrees around the selected TAZs.
bike = gpd.read_file("raw_data/LevelofTrafficStress4Cyclist_2023/LevelofTrafficStress4Cyclist_2023.shp")
bike = bike.to_crs(taz.crs)
bike = bike[bike.geometry.intersects(network_geometry)].copy()
bike.to_file("step1_data/bike.gpkg")

# Use this example's direction assumptions: B/T are two-way; F follows the geometry.
bike["directed"] = bike["DIRECTION"].map({"B": 0, "F": 1, "T": 0})

# Convert the mapped fields; table2gmns creates reverse rows for two-way features.
net = tg.getNetFromFile(bike, mode_types="bike", link_field_map={"geometry": "geometry", "directed": "directed"})
tg.fillLinkAttributesWithDefaultValues(net, default_speed=True, default_capacity=True) # The original SPEED and LANE fields describe motor vehicle speeds and lane counts.
tg.outputNetToCSV(net, output_folder="step2_networks/bike")

# Keep the largest strongly connected component of the exported bicycle network.
bike_node = pd.read_csv("step2_networks/bike/node.csv")
bike_link = pd.read_csv("step2_networks/bike/link.csv")
bike_graph = nx.from_pandas_edgelist(bike_link, "from_node_id", "to_node_id", create_using=nx.DiGraph())
largest_component = max(nx.strongly_connected_components(bike_graph), key=len)
bike_link = bike_link[bike_link["from_node_id"].isin(largest_component) & bike_link["to_node_id"].isin(largest_component)].copy()
bike_node = bike_node[bike_node["node_id"].isin(largest_component)].copy()

# Save the prepared bicycle network for this separate conversion example.
bike_node.to_csv("step2_networks/bike/node.csv", index=False)
bike_link.to_csv("step2_networks/bike/link.csv", index=False)


## Step 3. Zone Centroid Connectors Generation
### 3.1. Zone Centroid Generation
Run `extract_zones()` from `step3_zones/`; it reads `data/taz.shp` and writes `zone.csv`. `TAZ_ID` retains the source identifier; `zone_id` is the model identifier.


In [ ]:
# Place the zone shapefile in the data subfolder expected by extract_zones().
Path("step3_zones/data").mkdir(parents=True, exist_ok=True)
taz = gpd.read_file("step1_data/taz.gpkg")
taz = taz.to_crs("EPSG:4326")
taz.to_file("step3_zones/data/taz.shp")

# Generate zone.csv inside step3_zones, then return to the notebook folder.
with chdir("step3_zones"):
    gr.extract_zones()


**TAZ boundaries and centroids from `zone.csv`.** Labels show `zone_id`.


In [ ]:
# Read the TAZ polygons and generated zone points in the same metric CRS.
plot_taz = gpd.read_file("step1_data/taz.gpkg").to_crs("EPSG:26917")
plot_zones = pd.read_csv("step3_zones/zone.csv")
plot_zones = gpd.GeoDataFrame(plot_zones, geometry=gpd.GeoSeries.from_wkt(plot_zones["geometry"], crs="EPSG:4326"), crs="EPSG:4326").to_crs("EPSG:26917")

# Draw individual TAZ boundaries, the district outline and centroid points.
fig, ax = plt.subplots(figsize=(8, 8))
plot_taz.plot(ax=ax, facecolor="#F3F1F5", edgecolor="#999999", linewidth=0.65)
plot_taz.dissolve().boundary.plot(ax=ax, color="#222222", linewidth=1.2)
plot_zones.plot(ax=ax, color="#8056A3", markersize=25, edgecolor="white", linewidth=0.5, zorder=3)

# Label each point with its model zone_id.
for _, point in plot_zones.iterrows():
    ax.annotate(str(int(point["zone_id"])), (point.geometry.x, point.geometry.y), xytext=(4, 4), textcoords="offset points", fontsize=8)

# Add the legend and display the zone map.
ax.legend(handles=[Line2D([0], [0], color="#999999", lw=0.8, label="TAZ boundary"), Line2D([0], [0], marker="o", color="none", markerfacecolor="#8056A3", markersize=6, label="Generated zone centroid")], loc="lower left", frameon=True, framealpha=0.9)
ax.set_title(f"Step 3.1 | Zones and centroids ({len(plot_zones)} TAZs)", fontsize=14)
ax.set_aspect("equal")
ax.set_axis_off()
fig.tight_layout()
display(fig)
plt.close(fig)


### 3.2. Zone Centroid Connector Generation
Use `build_network()` with the Step 2 walking network and `zone.csv`. Dropping `boundary_geometry` enables point-based connector selection with the default 1,000-m search radius. Outputs are saved in `step3_zones/connected_walk_zone/`.


In [ ]:
# Attach zone centroids to the walking network and export the connected tables.
links, nodes, connectors = build_network(
    input_path="step2_networks/walk",
    output_path="step3_zones/connected_walk_zone",
    node_taz_df=pd.read_csv("step3_zones/zone.csv").drop(columns=["boundary_geometry"]),  # Use centroid points rather than zone boundaries.
)

# Set an illustrative capacity for zone connectors (default is 99999).
connectors["capacity"] = 3000
links.loc[links["link_type_name"] == "connector", "capacity"] = 3000

# Replace the default 5-mph rounding with an exact conversion from km/h.
links["vdf_free_speed_mph"] = links["free_speed"] / 1.609344
connectors["vdf_free_speed_mph"] = connectors["free_speed"] / 1.609344

connectors.to_csv("step3_zones/connected_walk_zone/connector_links.csv", index=False)
links.to_csv("step3_zones/connected_walk_zone/link.csv", index=False)

**Walking network with zone connectors and centroids highlighted.**


In [ ]:
# Read the exported network with zone connectors and project it for display.
plot_taz = gpd.read_file("step1_data/taz.gpkg").to_crs("EPSG:26917")
plot_nodes = pd.read_csv("step3_zones/connected_walk_zone/node.csv", low_memory=False)
plot_links = pd.read_csv("step3_zones/connected_walk_zone/link.csv", low_memory=False)
plot_links = gpd.GeoDataFrame(plot_links, geometry=gpd.GeoSeries.from_wkt(plot_links["geometry"], crs="EPSG:4326"), crs="EPSG:4326").to_crs("EPSG:26917")

# Identify zone nodes and all links touching those nodes.
plot_zones = plot_nodes[plot_nodes["zone_id"].notna()].copy()
zone_link = plot_links["from_node_id"].isin(plot_zones["node_id"]) | plot_links["to_node_id"].isin(plot_zones["node_id"])
plot_zones = gpd.GeoDataFrame(plot_zones, geometry=gpd.points_from_xy(plot_zones["x_coord"], plot_zones["y_coord"], crs="EPSG:4326")).to_crs("EPSG:26917")

# Draw physical walking links, zone connectors and centroid points separately.
fig, ax = plt.subplots(figsize=(8, 8))
plot_taz.boundary.plot(ax=ax, color="#D0D0D0", linewidth=0.5, zorder=0)
plot_links[~zone_link].plot(ax=ax, color="#3B79A5", linewidth=0.45, alpha=0.65)
plot_links[zone_link].plot(ax=ax, color="#8056A3", linewidth=1.2, zorder=3)
plot_zones.plot(ax=ax, color="#8056A3", markersize=14, edgecolor="white", linewidth=0.3, zorder=4)

# Show the directed connector count and display the map.
ax.set_title(f"Step 3.2 | Walking network with zone connections\n{zone_link.sum():,} directed connector links", fontsize=13)
ax.set_aspect("equal")
ax.set_axis_off()
fig.legend(handles=[Line2D([0], [0], color="#3B79A5", lw=1.3, label="Walking links"), Line2D([0], [0], color="#8056A3", lw=1.3, marker="o", markersize=4, label="Zone connectors / centroids")], loc="lower center", ncol=2, frameon=False)
fig.tight_layout(rect=(0, 0.05, 1, 1))
display(fig)
plt.close(fig)


## Step 4: Merge All Networks
### 4.1. Renumbering
Use `network_update()` to prevent node/link ID conflicts between the walking and transit networks.


In [ ]:
# Read the walking network with zones and the separate transit network.
base_node = pd.read_csv("step3_zones/connected_walk_zone/node.csv")
base_link = pd.read_csv("step3_zones/connected_walk_zone/link.csv")
transit_node = pd.read_csv("step2_networks/transit/node.csv")
transit_link = pd.read_csv("step2_networks/transit/link.csv")

# Renumber the networks so their node and link IDs can be combined.
base_node, merge_node, base_link, merge_link = network_update(base_node, base_link, transit_node, transit_link)

# Save the four renumbered tables for transfer generation and merging.
Path("step4_merge/connected_walk_zone_transit/renumbered").mkdir(parents=True, exist_ok=True)
base_node.to_csv("step4_merge/connected_walk_zone_transit/renumbered/base_node.csv", index=False)
merge_node.to_csv("step4_merge/connected_walk_zone_transit/renumbered/transit_node.csv", index=False)
base_link.to_csv("step4_merge/connected_walk_zone_transit/renumbered/base_link.csv", index=False)
merge_link.to_csv("step4_merge/connected_walk_zone_transit/renumbered/transit_link.csv", index=False)


### 4.2. Transfers
Use `transfer_connector_builder()` with `filter_node_type="stop"` and `search_radius=300`. Exclude zone nodes from the walking side.


In [ ]:
# Read renumbered nodes and exclude zone centroids from the walking side.
base_node = pd.read_csv("step4_merge/connected_walk_zone_transit/renumbered/base_node.csv")
merge_node = pd.read_csv("step4_merge/connected_walk_zone_transit/renumbered/transit_node.csv")
physical_base = base_node[base_node["zone_id"].isna()]

# Create transfers to physical transit stops within 300 m.
transfers = transfer_connector_builder(physical_base, merge_node, filter_node_type="stop", search_radius=300)

# Set an illustrative capacity for zone connectors (default is 100000).
transfers["capacity"] = 3000

# Save the transfer links for network integration.
transfers.to_csv("step4_merge/connected_walk_zone_transit/transfers.csv", index=False)


### 4.3. Network integration
Combine the renumbered networks and transfer links with `network_merge()`.


In [ ]:
# Read the four renumbered tables and the generated transfer links.
base_node = pd.read_csv("step4_merge/connected_walk_zone_transit/renumbered/base_node.csv")
merge_node = pd.read_csv("step4_merge/connected_walk_zone_transit/renumbered/transit_node.csv")
base_link = pd.read_csv("step4_merge/connected_walk_zone_transit/renumbered/base_link.csv")
merge_link = pd.read_csv("step4_merge/connected_walk_zone_transit/renumbered/transit_link.csv")
transfers = pd.read_csv("step4_merge/connected_walk_zone_transit/transfers.csv")

# Merge the networks and identify the base links as walking links.
node_all, link_all = network_merge(base_node, base_link, merge_node, merge_link, transfers, base_link_allowed_uses="walk")

# Keep endpoint IDs as integers after merging.
link_all = link_all.astype({"from_node_id": "int64", "to_node_id": "int64"})  

# Save the complete walk + transit network for validation and routing.
Path("step4_merge/connected_walk_zone_transit/connected_network").mkdir(parents=True, exist_ok=True)
node_all.to_csv("step4_merge/connected_walk_zone_transit/connected_network/node.csv", index=False)
link_all.to_csv("step4_merge/connected_walk_zone_transit/connected_network/link.csv", index=False)


**Merged walk + transit network.** The map is zoomed to Traffic District 92; the exported transit network is not clipped.


In [ ]:
# Set the map extent from the TAZ district, with a small display margin.
plot_taz = gpd.read_file("step1_data/taz.gpkg").to_crs("EPSG:26917")
plot_bounds = plot_taz.total_bounds
plot_margin = max(plot_bounds[2] - plot_bounds[0], plot_bounds[3] - plot_bounds[1]) * 0.06

# Create the map and read the merged GMNS network.
fig, ax = plt.subplots(figsize=(8, 8))
plot_nodes = pd.read_csv("step4_merge/connected_walk_zone_transit/connected_network/node.csv", low_memory=False)
plot_links = pd.read_csv("step4_merge/connected_walk_zone_transit/connected_network/link.csv", low_memory=False)
plot_links = gpd.GeoDataFrame(plot_links, geometry=gpd.GeoSeries.from_wkt(plot_links["geometry"], crs="EPSG:4326"), crs="EPSG:4326").to_crs("EPSG:26917")

# Classify links for plotting: zone connectors, transit, transfers and walking.
plot_zones = plot_nodes[plot_nodes["zone_id"].notna()].copy()
zone_link = plot_links["from_node_id"].isin(plot_zones["node_id"]) | plot_links["to_node_id"].isin(plot_zones["node_id"])
plot_transit_mask = plot_links["link_type_name"].isin(["service_links", "boarding_links", "deboarding_links", "transferring_links"])
plot_transfer_mask = plot_links["link_type_name"].eq("connector") & ~zone_link
plot_base_mask = ~(zone_link | plot_transit_mask | plot_transfer_mask)
plot_zones = gpd.GeoDataFrame(plot_zones, geometry=gpd.points_from_xy(plot_zones["x_coord"], plot_zones["y_coord"], crs="EPSG:4326")).to_crs("EPSG:26917")

# Draw each link category and overlay the zone nodes and district outline.
plot_links[plot_base_mask].plot(ax=ax, color="#3B79A5", linewidth=0.45, alpha=0.6)
plot_links[plot_transit_mask].plot(ax=ax, color="#C56E3D", linewidth=0.75, alpha=0.8)
plot_links[plot_transfer_mask].plot(ax=ax, color="#686868", linewidth=0.65, linestyle="--", alpha=0.7)
plot_links[zone_link].plot(ax=ax, color="#8056A3", linewidth=1.0, zorder=4)
plot_zones.plot(ax=ax, color="#8056A3", markersize=10, edgecolor="white", linewidth=0.3, zorder=5)
plot_taz.dissolve().boundary.plot(ax=ax, color="#222222", linewidth=1.0)

# Zoom the display to the district; leave the exported network unchanged.
ax.set_xlim(plot_bounds[0] - plot_margin, plot_bounds[2] + plot_margin)
ax.set_ylim(plot_bounds[1] - plot_margin, plot_bounds[3] + plot_margin)
ax.set_title("Step 4 | Walk + transit in Traffic District 92", fontsize=14)
ax.set_aspect("equal")
ax.set_axis_off()

# Add the legend, save the map and display it.
fig.legend(handles=[Line2D([0], [0], color="#3B79A5", lw=1.3, label="Walking"), Line2D([0], [0], color="#C56E3D", lw=1.3, label="Transit"), Line2D([0], [0], color="#686868", lw=1.0, linestyle="--", label="Base-to-transit transfers"), Line2D([0], [0], color="#8056A3", lw=1.3, label="Zone connectors")], loc="lower center", ncol=2, frameon=False)
fig.tight_layout(rect=(0, 0.08, 1, 1))
fig.savefig("step4_merge/merged_networks.png", dpi=180, bbox_inches="tight")
display(fig)
plt.close(fig)


## Step 5: Network Validation
Run `validate_network()` from `step4_merge/connected_walk_zone_transit/`. It checks the files in `connected_network/` and saves `network_validation_report.json` there.


In [ ]:
# Validate the node/link files in connected_network and save the JSON report.
with chdir("step4_merge/connected_walk_zone_transit"):
    gr.validate_network()


**Additional validation: `quick_check()`.** Create the illustrative OD demand first, then pass the folder containing `node.csv`, `link.csv` and `demand.csv`. The function checks IDs, attributes, zone connectors and demand references, and saves `validation_report.txt` and `zone_connector_report.csv` without modifying the input tables.

Connected-component analysis is not implemented in this check. Warnings about missing routing outputs are expected before Step 6; the speed-rounding check also flags the 3-mph walking speed.

## Step 6: Accessibility analysis and interpretation of results
### 6.1. Routing and Accessibility Analysis
Run `DTALite.assignment()` for 14:00–16:00 under two settings:

- **Free flow:** `number_of_iterations=0`, using stored link travel times, including transit and boarding times.
- **Demand and congestion:** `number_of_iterations=40`, assigning `demand.csv` and updating travel times using capacities and VDF parameters. Iterations control the calculation, not the demand level; convergence is not guaranteed.

Results are saved under `step6_accessibility/freeflow/` and `step6_accessibility/congestion/`. Separate Python processes prevent DTALite state from carrying over between runs.

`base_demand_mode=0` disables baseline demand, not the supplied `demand.csv`. `route_output=1` requests assigned-route output.


In [ ]:
# Run each DTALite case in a fresh process using the notebook's Python environment.
import subprocess
import sys


#### 6.1.1. Free-flow travel times


In [ ]:
# Copy the merged network into a separate folder for the free-flow case.
Path("step6_accessibility/freeflow/walk_transit/connected_network").mkdir(parents=True, exist_ok=True)
shutil.copy2("step4_merge/connected_walk_zone_transit/connected_network/node.csv", "step6_accessibility/freeflow/walk_transit/connected_network/node.csv")
shutil.copy2("step4_merge/connected_walk_zone_transit/connected_network/link.csv", "step6_accessibility/freeflow/walk_transit/connected_network/link.csv")

# Use zero iterations to calculate OD travel times without volume-dependent assignment.
settings = pd.DataFrame([{"number_of_iterations": 0, "demand_period_starting_hours": 14, "demand_period_ending_hours": 16, "base_demand_mode": 0, "route_output": 1}])
settings.to_csv("step6_accessibility/freeflow/walk_transit/connected_network/settings.csv", index=False)

# Start a fresh DTALite process so previous runs cannot carry over engine state.
run = subprocess.run([sys.executable, "-c", "import DTALite as dta; dta.assignment()"], cwd="step6_accessibility/freeflow/walk_transit/connected_network", capture_output=True, text=True, check=True)
print(run.stdout)

#### 6.1.2. Demand- and congestion-dependent travel times
Use **100 demand units per ordered OD pair**, totaling 87,000 across 30 TAZs. Transit-link capacity is 3,000; other capacities and VDF parameters are retained. These are illustrative assumptions, not a calibrated pedestrian or transit-crowding model.


In [ ]:
# Copy the same network and the OD demand into the congestion-case folder.
Path("step6_accessibility/congestion/walk_transit/connected_network").mkdir(parents=True, exist_ok=True)
shutil.copy2("step4_merge/connected_walk_zone_transit/connected_network/node.csv", "step6_accessibility/congestion/walk_transit/connected_network/node.csv")
shutil.copy2("step4_merge/connected_walk_zone_transit/connected_network/link.csv", "step6_accessibility/congestion/walk_transit/connected_network/link.csv")
shutil.copy2("step6_accessibility/congestion/demand.csv", "step6_accessibility/congestion/walk_transit/connected_network/demand.csv")

# Run 40 assignment iterations with the supplied demand, capacities and VDF parameters.
settings = pd.DataFrame([{"number_of_iterations": 40, "demand_period_starting_hours": 14, "demand_period_ending_hours": 16, "base_demand_mode": 0, "route_output": 1}])
settings.to_csv("step6_accessibility/congestion/walk_transit/connected_network/settings.csv", index=False)

# Start a fresh DTALite process, independently of the free-flow run.
run = subprocess.run([sys.executable, "-c", "import DTALite as dta; dta.assignment()"], cwd="step6_accessibility/congestion/walk_transit/connected_network", capture_output=True, text=True, check=True)
print(run.stdout)

**Additional validation: `quick_check()`.** This function checks file structure, IDs, attributes, zone connectors and available demand references. The congestion-case folder is used here because it contains the assignment outputs. For these outputs, the check confirms file availability but does not evaluate travel times, detours or route reasonableness. Passing this check therefore does not establish the validity of the analysis results.

In [ ]:
check_results = quick_check(input_dir="step6_accessibility/congestion/walk_transit/connected_network")

# Warnings in this tutorial
# [3A] Speeds are not rounded to 5 km/h. This is intentional; no correction is needed.
# [7B] `odme_mode` is unspecified. OD demand estimation is not used here; setting it to `0` clears the warning.

### 6.2. Post-Processing Analysis
Count other TAZs reachable within **30 minutes**, using `total_free_flow_travel_time` for free flow and `total_congestion_travel_time` for assignment. Retain origins with zero reachable destinations.


In [ ]:
# Prepare one accessibility summary per travel-time case.
zone = pd.read_csv("step3_zones/zone.csv")
summaries = []

# Use the corresponding travel-time field from each DTALite output.
for case, time_column in [("freeflow", "total_free_flow_travel_time"), ("congestion", "total_congestion_travel_time")]:
    # Read interzonal OD results and check that reported travel times are usable.
    od = pd.read_csv(f"step6_accessibility/{case}/walk_transit/connected_network/od_performance.csv")
    od = od[od["o_zone_id"] != od["d_zone_id"]].copy()
    assert not od.empty, "No interzonal OD results were generated."
    assert od[time_column].notna().all() and od[time_column].ge(0).all(), "Missing or negative OD travel times."

    # Count distinct destinations reachable within 30 minutes.
    reachable = od[od[time_column] <= 30]
    counts = reachable.groupby("o_zone_id")["d_zone_id"].nunique()

    # Keep every origin TAZ, including those with zero reachable destinations.
    case_summary = zone[["zone_id", "TAZ_ID"]].copy()
    case_summary["reachable_TAZs"] = case_summary["zone_id"].map(counts).fillna(0).astype(int)
    case_summary["mean_travel_time_min"] = case_summary["zone_id"].map(od.groupby("o_zone_id")[time_column].mean())
    case_summary["scenario"] = "walk_transit"
    case_summary["case"] = case
    case_summary["time_limit_min"] = 30
    summaries.append(case_summary)

# Combine the cases and save the per-TAZ accessibility table.
summary = pd.concat(summaries, ignore_index=True)
summary.to_csv("step6_accessibility/accessibility_summary.csv", index=False)

# Compare reachable-TAZ counts and mean reported travel times across cases.
display(summary.groupby("case", sort=False)["reachable_TAZs"].agg(["min", "median", "mean", "max"]))
print("Mean reported OD travel time (minutes)")
display(summary.groupby("case", sort=False)["mean_travel_time_min"].mean())
